# ÉLUME AI Hair Studio — Kaggle GPU Backend v3
> Run cells **1 → 8** in order. Keep **Cell 8** running during demo.

---

### Pre-flight checklist
| Step | Where | Value |
|---|---|---|
| Accelerator | Session options | **GPU T4 x2** |
| Internet | Session options | **On** |
| Persistence | Session options | **Files Only** |
| Dataset | Add-ons › Datasets | `acetrading/elume-sdxl-inpaint` |
| Secrets | Add-ons › Secrets | See below |

### Kaggle Secrets
| Label | Where to get it |
|---|---|
| `NGROK_TOKEN` | dashboard.ngrok.com/get-started/your-authtoken |
| `NGROK_DOMAIN` | dashboard.ngrok.com/domains (your free static domain) |
| `GROQ_API_KEY` | console.groq.com |

---

### API Endpoints
| Endpoint | Method | Description |
|---|---|---|
| `/health` | GET | GPU + VRAM status |
| `/analyse` | POST | Face metrics → personalised style cards (Groq Vision) |
| `/generate` | POST | Photo + prompt → SDXL hair inpaint |
| `/mask_preview` | POST | Hair mask preview – fast, no SDXL |
| `/describe_style` | POST | Photo or text → SDXL prompt |
| `/consult` | POST | Style → full maintenance consultation |


In [ ]:
# Cell 1 – Install packages
# ──────────────────────────────────────────────────────────────────────────
# WHY the pre-uninstall + sys.modules eviction:
#   Kaggle ships old diffusers/accelerate/peft in /usr/local/lib/python3.12/
#   dist-packages/ AND pre-loads them into sys.modules before Cell 1 runs.
#
#   Problem A (disk):  pip install without uninstall first layers the new
#     version under the old one in sys.path priority, so Python still imports
#     the stale files.
#
#   Problem B (memory): even after fixing the disk, Python finds the stale
#     module objects already in sys.modules and never touches the new files.
#     importlib.invalidate_caches() only clears FileFinder directory listings
#     — it does NOT evict sys.modules entries.
#
#   Fix: uninstall (disk), reinstall (disk), evict sys.modules (memory),
#     invalidate_caches (FileFinder), then verify with smoke-test imports.
# ──────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib

# ── Step 1: evict stale copies from disk ──────────────────────────────────
_PREUNINSTALL = ["diffusers", "accelerate", "peft"]
print("Pre-uninstalling stale packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y"] + _PREUNINSTALL,
    check=False,
    capture_output=True,
)
print("  done")

# ── Step 2: install pinned versions ───────────────────────────────────────
# Pinning strategy:
#   diffusers:            <0.32.0 keeps stable pipeline API
#   accelerate:           >=0.33.0 required – clear_device_cache added there;
#                         peft>=0.11 imports it unconditionally
#   peft:                 explicit pin so it resolves consistently with accel
#   fastapi/uvicorn/starlette/transformers: loose minimums so Kaggle's
#     pre-installed google-adk, gradio, mcp don't conflict
PKGS = [
    "diffusers>=0.28.2,<0.32.0",
    "transformers>=4.41.0,<5.0",
    "accelerate>=0.33.0,<1.0",
    "peft>=0.11.0,<2.0",
    "safetensors",
    "fastapi>=0.115.2,<1.0",
    "uvicorn[standard]>=0.31.1,<1.0",
    "starlette>=0.40.0,<1.0",
    "pyngrok==7.1.6",
    "nest_asyncio",
    "python-multipart",
    "opencv-python-headless",
    "gdown",
    "xformers",
    "groq",
    "Pillow>=10.0",
    "pillow-heif",
    "requests",
]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + PKGS,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("pip install failed")
print("Packages installed")

# ── Step 3: evict stale in-memory module objects ──────────────────────────
_EVICT_PREFIXES = ("diffusers", "accelerate", "peft")
_evicted = [
    k for k in list(sys.modules)
    if any(k == p or k.startswith(p + ".") for p in _EVICT_PREFIXES)
]
for k in _evicted:
    del sys.modules[k]
importlib.invalidate_caches()
print(f"Evicted {len(_evicted)} stale sys.modules entries "
      f"({', '.join(_EVICT_PREFIXES)})")

# ── Step 4: sanity checks ─────────────────────────────────────────────────
import diffusers
from packaging.version import Version

_ver = Version(diffusers.__version__)
assert Version("0.28.2") <= _ver < Version("0.32.0"), \
    f"diffusers version out of range: {diffusers.__version__}"

# Smoke-test the exact symbol that was originally broken.
from diffusers.utils.constants import SAFE_WEIGHTS_INDEX_NAME          # noqa

import accelerate
assert Version(accelerate.__version__) >= Version("0.33.0"), \
    f"accelerate too old: {accelerate.__version__} — clear_device_cache needs >=0.33.0"

import peft

# Full pipeline import — exercises the entire diffusers → peft → accelerate chain.
from diffusers import StableDiffusionXLInpaintPipeline                 # noqa

print(f"\u2713 diffusers       {diffusers.__version__}")
print(f"\u2713 accelerate      {accelerate.__version__}")
print(f"\u2713 peft            {peft.__version__}")
print(f"\u2713 SAFE_WEIGHTS_INDEX_NAME importable")
print(f"\u2713 StableDiffusionXLInpaintPipeline importable")

import torch
print(f"\u2713 PyTorch {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found – set Accelerator to GPU T4 x2")


In [ ]:
# Cell 2 – Load Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from groq import Groq

def _require(s, key):
    val = s.get_secret(key)
    if not val:
        raise ValueError(f"Secret '{key}' is empty – add it in Add-ons > Secrets")
    return val

try:
    _s = UserSecretsClient()
    NGROK_TOKEN  = _require(_s, "NGROK_TOKEN")
    NGROK_DOMAIN = _require(_s, "NGROK_DOMAIN")
    GROQ_API_KEY = _require(_s, "GROQ_API_KEY")
    print("All secrets loaded")
    print(f"  ngrok : https://{NGROK_DOMAIN}")
    print(f"  groq  : {GROQ_API_KEY[:8]}...")
except Exception as e:
    raise RuntimeError(str(e))

groq_client = Groq(api_key=GROQ_API_KEY)

def groq_text(prompt, model="llama-3.3-70b-versatile", max_tokens=2500):
    res = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return res.choices[0].message.content

print("Groq client ready")
print("  Text  : llama-3.3-70b-versatile")
print("  Vision: meta-llama/llama-4-scout-17b-16e-instruct")


In [ ]:
# Cell 3 – BiSeNet hair-segmentation weights
import os, sys, shutil, gdown

BISENET_REPO    = "face-parsing.PyTorch"
BISENET_WEIGHTS = f"{BISENET_REPO}/res/cp/79999_iter.pth"
WEIGHTS_MIN_MB  = 40
DATASET         = "/kaggle/input/elume-sdxl-inpaint"

def _file_ok(path, min_mb):
    return os.path.exists(path) and os.path.getsize(path) >= min_mb * 1_000_000

if not os.path.exists(BISENET_REPO):
    print("Cloning BiSeNet...")
    os.system("git clone -q https://github.com/zllrunning/face-parsing.PyTorch")
else:
    print("BiSeNet repo present")

os.makedirs(f"{BISENET_REPO}/res/cp", exist_ok=True)

# ── Try dataset copy first (fast), then fall back to gdown ───────────────
_DATASET_BISENET = f"{DATASET}/bisenet_79999_iter.pth"
if os.path.exists(_DATASET_BISENET) and not _file_ok(BISENET_WEIGHTS, WEIGHTS_MIN_MB):
    shutil.copy(_DATASET_BISENET, BISENET_WEIGHTS)
    print(f"BiSeNet weights loaded from dataset ({os.path.getsize(BISENET_WEIGHTS)/1e6:.0f} MB)")

for attempt in range(1, 4):
    if _file_ok(BISENET_WEIGHTS, WEIGHTS_MIN_MB):
        break
    print(f"Downloading BiSeNet weights (attempt {attempt}/3)...")
    try:
        gdown.download(
            "https://drive.google.com/uc?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812",
            BISENET_WEIGHTS, quiet=False
        )
    except Exception as e:
        print(f"  Attempt {attempt} failed: {e}")
else:
    if not _file_ok(BISENET_WEIGHTS, WEIGHTS_MIN_MB):
        raise RuntimeError("BiSeNet download failed after 3 attempts")

sys.path.insert(0, f"/kaggle/working/{BISENET_REPO}")
print(f"BiSeNet ready – {os.path.getsize(BISENET_WEIGHTS)/1e6:.0f} MB")


In [ ]:
# Cell 4 – Load BiSeNet + SDXL Inpainting
import gc, os, sys, zipfile, shutil, torch
from pathlib import Path
from diffusers import StableDiffusionXLInpaintPipeline
from torchvision import transforms
from model import BiSeNet  # from face-parsing.PyTorch (added to sys.path in Cell 3)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/kaggle/working/sdxl-model"
# DATASET defined in Cell 3

def _free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _vram_str():
    if not torch.cuda.is_available():
        return "CPU"
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    return f"{used:.1f}/{total:.1f} GB"

# ── Restore ResNet18 backbone from dataset (avoids re-download) ───────────
_RN18_DATASET = f"{DATASET}/resnet18-5c106cde.pth"
_RN18_CACHE   = Path("/root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth")
if os.path.exists(_RN18_DATASET) and not _RN18_CACHE.exists():
    _RN18_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(_RN18_DATASET, str(_RN18_CACHE))
    print("ResNet18 backbone restored from dataset")

# ── Resolve SDXL model path ───────────────────────────────────────────────
# Priority 1: model_index.json directly in dataset root (flat layout)
# Priority 2: model_index.json inside sdxl-inpaint/ subfolder
# Priority 3: already extracted to working dir
# Priority 4: fall back to HuggingFace download
def _find_model_index(base):
    """Return directory containing model_index.json, or None."""
    if os.path.exists(os.path.join(base, "model_index.json")):
        return base
    for item in os.listdir(base):
        c = os.path.join(base, item)
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "model_index.json")):
            return c
    return None

# Check dataset first — if model_index.json is already there, use it directly
# (no copying needed — Kaggle datasets are read-only but diffusers can load from them)
_dataset_model = _find_model_index(DATASET) if os.path.exists(DATASET) else None

if _dataset_model:
    MODEL_PATH = _dataset_model
    print(f"SDXL model found in dataset: {MODEL_PATH}")
elif os.path.exists(os.path.join("/kaggle/working/sdxl-model", "model_index.json")):
    MODEL_PATH = "/kaggle/working/sdxl-model"
    print("SDXL model already extracted to working dir")
else:
    print("Dataset not found – falling back to HuggingFace download (slow)")
    MODEL_PATH = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

# ── BiSeNet ────────────────────────────────────────────────────────────────
print("Loading BiSeNet...")
bisenet = BiSeNet(n_classes=19)
bisenet.load_state_dict(torch.load(BISENET_WEIGHTS, map_location=DEVICE))
bisenet.to(DEVICE).eval()
_free_vram()
print(f"BiSeNet loaded | VRAM: {_vram_str()}")

# ── SDXL ───────────────────────────────────────────────────────────────────
print(f"Loading SDXL from {MODEL_PATH}...")
_local = not MODEL_PATH.startswith("diffusers/")
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True,
    local_files_only=_local,
)
# CPU offload keeps VRAM free between calls – essential for T4 15 GB
pipe.enable_model_cpu_offload()
try:
    pipe.enable_xformers_memory_efficient_attention()
    ATTN_MODE = "xformers"
except Exception:
    pipe.enable_attention_slicing(1)
    ATTN_MODE = "attention_slicing"
_free_vram()
print(f"SDXL loaded ({ATTN_MODE}) | VRAM: {_vram_str()}")
print("All models ready")


In [ ]:
# Cell 4b – Save downloaded models to Kaggle dataset (run ONCE after first download)
# Run this immediately after Cell 4 completes on a session where models were
# freshly downloaded (i.e. the dataset was empty or missing).
# Future sessions will skip all downloads and load from the dataset instead.
import os, shutil, glob, json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/elume-sdxl-inpaint")
SAVE_DIR.mkdir(exist_ok=True)

# ── SDXL: locate HuggingFace snapshot and copy ────────────────────────────
HF_CACHE = Path("/root/.cache/huggingface/hub")
pattern  = "models--diffusers--stable-diffusion-xl-1.0-inpainting-0.1/snapshots/*"
snapshots = sorted(glob.glob(str(HF_CACHE / pattern)))
if not snapshots:
    raise RuntimeError("HF snapshot not found – did Cell 4 complete successfully?")

HF_SNAPSHOT = Path(snapshots[-1])
size_gb = sum(f.stat().st_size for f in HF_SNAPSHOT.rglob("*") if f.is_file()) / 1e9
print(f"HF snapshot : {HF_SNAPSHOT}")
print(f"Size        : {size_gb:.1f} GB")

SDXL_DST = SAVE_DIR / "sdxl-inpaint"
if SDXL_DST.exists():
    print("sdxl-inpaint/ already copied – skipping")
else:
    print("Copying SDXL model files (this takes ~2 min on T4 NVMe)...")
    shutil.copytree(HF_SNAPSHOT, SDXL_DST, symlinks=False)
    print("  done")

# ── BiSeNet weights ───────────────────────────────────────────────────────
BISENET_SRC = Path("/kaggle/working/face-parsing.PyTorch/res/cp/79999_iter.pth")
BISENET_DST = SAVE_DIR / "bisenet_79999_iter.pth"
if BISENET_SRC.exists():
    shutil.copy(BISENET_SRC, BISENET_DST)
    print(f"Copied BiSeNet weights  ({BISENET_SRC.stat().st_size/1e6:.0f} MB)")
else:
    print("WARNING: BiSeNet weights not found – re-run Cell 3 first")

# ── ResNet18 backbone (BiSeNet dependency) ────────────────────────────────
RN18_SRC = Path("/root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth")
RN18_DST = SAVE_DIR / "resnet18-5c106cde.pth"
if RN18_SRC.exists():
    shutil.copy(RN18_SRC, RN18_DST)
    print(f"Copied ResNet18 backbone ({RN18_SRC.stat().st_size/1e6:.0f} MB)")
else:
    print("WARNING: ResNet18 not found at expected path – BiSeNet may re-download it next session")

# ── Dataset metadata ──────────────────────────────────────────────────────
(SAVE_DIR / "dataset-metadata.json").write_text(json.dumps({
    "title": "ELUME SDXL Inpaint",
    "id":    "acetrading/elume-sdxl-inpaint",
    "licenses": [{"name": "other"}]
}, indent=2))

# ── Verify before upload ──────────────────────────────────────────────────
total = sum(f.stat().st_size for f in SAVE_DIR.rglob("*") if f.is_file()) / 1e9
print(f"\nTotal staged: {total:.1f} GB")
print("Contents:")
for p in sorted(SAVE_DIR.iterdir()):
    if p.is_dir():
        sz = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
        print(f"  {p.name}/  ({sz:.1f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size/1e6:.0f} MB)")

# ── Upload ────────────────────────────────────────────────────────────────
print("\nUploading to acetrading/elume-sdxl-inpaint ...")
print("(13+ GB – expect 5–15 min depending on Kaggle upload speed)")
ret = os.system(f"kaggle datasets version -p {SAVE_DIR} -m 'SDXL + BiSeNet + ResNet18' --dir-mode zip")
if ret == 0:
    print("\nDataset updated. Future sessions will load from dataset, no downloads needed.")
else:
    print(f"\nkaggle CLI returned {ret} – check output above for errors")


In [ ]:
# Cell 5 – Hair pipeline (mask, composite, generate)
import cv2, io, base64, numpy as np, torch
from PIL import Image, ImageOps
from torchvision import transforms
from typing import Optional
import pillow_heif
pillow_heif.register_heif_opener()

BISENET_TRANSFORM = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

NEGATIVE_PROMPT = (
    "wig, toupee, fake hair, unnatural hairline, plastic, deformed, bad anatomy,"
    " blurry, low quality, watermark, cartoon, drawing, painting, anime,"
    " helmet, hat, bald patches, disconnected hair, floating strands,"
    " jpeg artifacts, overexposed, duplicate"
)

CLIP_WORD_LIMIT = 50  # SDXL CLIP tokeniser limit guard


def load_image(data: bytes) -> Image.Image:
    img = Image.open(io.BytesIO(data))
    img = ImageOps.exif_transpose(img)   # fix iPhone rotation
    return img.convert("RGB")


def get_hair_mask(img_pil: Image.Image, dilation_px: int = 28) -> Image.Image:
    w, h = img_pil.size
    tensor = BISENET_TRANSFORM(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = bisenet(tensor)[0]
    parsing = logits.squeeze(0).cpu().numpy().argmax(0)
    mask = np.zeros(parsing.shape, dtype=np.uint8)
    for lbl in [17, 18]:   # 17=hair, 18=crown/hat area
        mask[parsing == lbl] = 255
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    # Close holes in curly/coiled hair BEFORE dilating
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px, dilation_px))
    mask = cv2.dilate(mask, k, iterations=2)
    mask = cv2.GaussianBlur(mask, (31, 31), 0)
    _, mask = cv2.threshold(mask, 30, 255, cv2.THRESH_BINARY)
    return Image.fromarray(mask)


def paste_face_back(
    original:  Image.Image,
    generated: Image.Image,
    hair_mask: Image.Image,
) -> Image.Image:
    w, h = original.size
    generated = generated.resize((w, h), Image.LANCZOS)
    hair_mask = hair_mask.resize((w, h), Image.LANCZOS)
    orig_arr  = np.array(original.convert("RGB"))
    gen_arr   = np.array(generated.convert("RGB"))
    mask_f    = np.array(hair_mask.convert("L")).astype(np.float32) / 255.0
    # Erode + blur the FACE region for soft feathered composite
    face_u8     = ((1.0 - mask_f) * 255).astype(np.uint8)
    ek          = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    face_eroded = cv2.erode(face_u8, ek, iterations=2)
    face_f      = cv2.GaussianBlur(face_eroded.astype(np.float32), (21, 21), 0) / 255.0
    face_f      = face_f[..., np.newaxis]
    composited  = (orig_arr * face_f + gen_arr * (1.0 - face_f)).astype(np.uint8)
    return Image.fromarray(composited)


def _clip_guard(prompt: str) -> str:
    words = prompt.split()
    return " ".join(words[:CLIP_WORD_LIMIT]) if len(words) > CLIP_WORD_LIMIT else prompt


def generate_hair(
    user_photo:      Image.Image,
    prompt:          str,
    seed:            int   = 42,
    steps:           int   = 30,
    guidance:        float = 8.0,
    strength:        float = 0.95,
    dilation:        int   = 28,
    reference_image: Optional[Image.Image] = None,
) -> tuple:
    w, h  = user_photo.size
    scale = min(1024 / max(w, h), 1.0)
    new_w = int((w * scale) // 8) * 8
    new_h = int((h * scale) // 8) * 8
    img   = user_photo.resize((new_w, new_h), Image.LANCZOS).convert("RGB")
    hair_mask = get_hair_mask(img, dilation_px=dilation)

    final_prompt = prompt
    if reference_image is not None:
        ref = reference_image.resize((256, 256))
        arr = np.array(ref.convert("RGB"))
        avg = arr.mean(axis=(0, 1)).astype(int)
        std = float(arr.std())
        tone = "dark" if arr.mean() < 80 else ("medium" if arr.mean() < 160 else "light")
        tex  = "curly" if std > 60 else ("wavy" if std > 35 else "straight")
        final_prompt = (
            f"{prompt}, hair color RGB({avg[0]},{avg[1]},{avg[2]}),"
            f" {tex} texture, {tone} tone, professional salon quality"
        )
    final_prompt = _clip_guard(final_prompt)
    print(f"Generating | seed={seed} steps={steps} | {final_prompt[:80]}...")

    generator = torch.Generator(device="cpu").manual_seed(seed)
    result = pipe(
        prompt              = final_prompt,
        negative_prompt     = NEGATIVE_PROMPT,
        image               = img,
        mask_image          = hair_mask,
        guidance_scale      = guidance,
        num_inference_steps = steps,
        strength            = strength,
        generator           = generator,
    ).images[0]

    composited = paste_face_back(img, result, hair_mask)
    if scale < 1.0:
        composited = composited.resize((w, h), Image.LANCZOS)
        hair_mask  = hair_mask.resize((w, h), Image.NEAREST)
    return composited, hair_mask


# Smoke-test
_dummy = Image.fromarray(np.random.randint(80, 200, (256, 256, 3), dtype=np.uint8))
assert get_hair_mask(_dummy).size == _dummy.size
print("Hair pipeline ready")


In [ ]:
# Cell 6 – Face analysis engine
# Converts raw MediaPipe numbers into specific style rules and constraints.
# This forces genuinely different recommendations per face.

def build_face_profile(face_shape, face_length, jaw_width,
                       forehead_width, cheek_width, ratio):
    try:
        jaw   = float(jaw_width)
        fore  = float(forehead_width)
        cheek = float(cheek_width)
        rat   = float(ratio)
    except (ValueError, TypeError) as e:
        return {"face_shape": face_shape, "rules": [], "avoid": [],
                "embrace": [], "sdxl_modifiers": [], "error": str(e)}

    jf  = jaw / fore  if fore  > 0 else 1.0   # jaw/forehead ratio
    cj  = cheek / jaw  if jaw  > 0 else 1.0   # cheek/jaw prominence
    cf  = cheek / fore if fore > 0 else 1.0   # cheek/forehead

    rules, avoid, embrace, sdxl_mod = [], [], [], []

    # ── Elongation ────────────────────────────────────────────────────────
    if rat > 1.8:
        rules.append(f"Very long face (ratio={rat:.2f}): must add width not height")
        avoid.extend(["tall updos", "high ponytails", "stacked crown volume"])
        embrace.extend(["blunt fringe", "curtain bangs", "wide fluffy sides", "bixie"])
        sdxl_mod.extend(["wide horizontal silhouette", "voluminous sides"])
    elif rat > 1.55:
        rules.append(f"Long face (ratio={rat:.2f}): favour width over length")
        avoid.extend(["centre-parted sleek straight hair that falls past collarbone"])
        embrace.extend(["side-swept bangs", "layered outward flicks", "lob with waves"])
        sdxl_mod.extend(["outward-flicked layers", "side-swept"])
    elif rat < 1.1:
        rules.append(f"Very wide/short face (ratio={rat:.2f}): must add height")
        avoid.extend(["jaw-level bobs that add width", "blunt full fringe", "wide sides"])
        embrace.extend(["height at crown", "long V-cut layers", "high bun", "pompadour"])
        sdxl_mod.extend(["crown height", "elongating V-cut layers"])
    elif rat < 1.3:
        rules.append(f"Moderately short face (ratio={rat:.2f}): light height helps")
        embrace.extend(["loose updo", "soft layers with root lift"])
        sdxl_mod.extend(["root lift", "airy layers"])
    else:
        rules.append(f"Balanced proportions (ratio={rat:.2f}): most lengths work")

    # ── Jaw vs forehead ───────────────────────────────────────────────────
    if jf > 1.18:
        rules.append(f"Jaw-dominant face (jaw/fore={jf:.2f}): widen forehead, slim jaw")
        avoid.extend(["jaw-length one-length bobs", "sharp lines at jaw", "heavy jaw volume"])
        embrace.extend(["volume at temples", "layers above jaw", "soft top-heavy silhouette"])
        sdxl_mod.extend(["volume at temples and crown", "tapered toward jaw"])
    elif jf < 0.82:
        rules.append(f"Forehead-dominant face (jaw/fore={jf:.2f}): cover forehead, add chin width")
        avoid.extend(["centre-parted sleek styles with exposed forehead", "slicked back"])
        embrace.extend(["side-swept fringe", "brow-grazing bangs", "chin-level flicks"])
        sdxl_mod.extend(["fringe at brow level", "chin-level outward flick"])
    else:
        rules.append(f"Balanced jaw-forehead (jaw/fore={jf:.2f}): strong versatility")

    # ── Cheekbones ────────────────────────────────────────────────────────
    if cj > 1.18 and cf > 1.1:
        rules.append(f"Prominent cheekbones (cheek/jaw={cj:.2f}): can handle any silhouette")
        embrace.extend(["geometric blunt cuts", "sculpted bobs", "structured styles"])
        sdxl_mod.extend(["geometric structured silhouette"])
    elif cj < 0.9 and cf < 0.9:
        rules.append(f"Recessed cheekbones (cheek/jaw={cj:.2f}): add midface width")
        embrace.extend(["layers flicked at cheekbone level", "feathered midlength"])
        sdxl_mod.extend(["flicked layers at cheekbone height"])

    # ── Named shape overrides ─────────────────────────────────────────────
    shape = face_shape.lower()
    if "oval" in shape:
        rules.append("Oval: benchmark face – can carry adventurous styles")
        embrace.extend(["bold geometric", "textured shag", "strong asymmetric"])
    elif "round" in shape:
        rules.append("Round: elongate and narrow visual width")
        avoid.extend(["chin-length blunt bobs", "round layers", "center parts"])
        embrace.extend(["long layers", "asymmetric cuts", "off-centre parts", "V-cut"])
        sdxl_mod.extend(["elongating asymmetric layers", "V-shaped hemline"])
    elif "square" in shape:
        rules.append("Square: soften jaw angles, add height")
        avoid.extend(["blunt jaw-length cuts", "severe straight fringe"])
        embrace.extend(["soft waves", "curved layers", "curtain bangs", "undulating texture"])
        sdxl_mod.extend(["soft waves", "curved flowing layers"])
    elif "heart" in shape or "inverted" in shape:
        rules.append("Heart: balance wide forehead with chin fullness")
        avoid.extend(["wide crown volume", "short pixie with flat sides"])
        embrace.extend(["chin-length lob with flicks", "side-swept bangs", "jaw-level waves"])
        sdxl_mod.extend(["chin-level fullness", "side-swept"])
    elif "diamond" in shape:
        rules.append("Diamond: widen forehead and chin, soften cheek prominence")
        avoid.extend(["maximum cheekbone-width styles"])
        embrace.extend(["brow fringe", "chin flicks", "textured ends that add chin width"])
        sdxl_mod.extend(["fringe at brow level", "textured chin-length ends"])
    elif "oblong" in shape or "rectangle" in shape:
        rules.append("Oblong/rectangle: add width, strongly avoid adding height")
        avoid.extend(["long straight centre-parted hair", "high updos"])
        embrace.extend(["waves", "curls", "blunt bangs", "layered lob"])
        sdxl_mod.extend(["wavy texture", "horizontal volume at sides"])

    return {
        "face_shape": face_shape,
        "metrics": {
            "jaw_fore_ratio":   round(jf, 3),
            "cheek_jaw_ratio":  round(cj, 3),
            "cheek_fore_ratio": round(cf, 3),
            "elongation_ratio": round(rat, 3),
        },
        "rules":          rules,
        "avoid":          list(set(avoid)),
        "embrace":        list(set(embrace)),
        "sdxl_modifiers": list(set(sdxl_mod)),
    }


def profile_to_prompt_block(p: dict) -> str:
    lines = [
        "FACE ANALYSIS (hard constraints – you MUST follow these):",
        f"  Shape: {p['face_shape']}",
    ]
    m = p.get("metrics", {})
    if m:
        lines.append(
            f"  Elongation ratio: {m.get('elongation_ratio','?')}  "
            f"jaw/fore: {m.get('jaw_fore_ratio','?')}  "
            f"cheek/jaw: {m.get('cheek_jaw_ratio','?')}"
        )
    if p.get("rules"):
        lines.append("Diagnostic rules:")
        for r in p["rules"]:
            lines.append(f"  * {r}")
    if p.get("avoid"):
        lines.append("AVOID (these will worsen the face proportions):")
        for a in p["avoid"]:
            lines.append(f"  - {a}")
    if p.get("embrace"):
        lines.append("EMBRACE (these will flatter this face):")
        for e in p["embrace"]:
            lines.append(f"  + {e}")
    if p.get("sdxl_modifiers"):
        lines.append(f"Required SDXL modifiers: {', '.join(p['sdxl_modifiers'])}")
    return "\n".join(lines)


print("Face analysis engine ready")
# Quick unit test
_p = build_face_profile("oval", "1.6", "0.85", "0.95", "1.0", "1.65")
assert _p["metrics"]["jaw_fore_ratio"] == round(0.85 / 0.95, 3)
print(f"  Test profile: {_p['face_shape']} | rules: {len(_p['rules'])}")


In [ ]:
# Cell 7 – FastAPI server definition (does NOT start serving yet)
import os, io, base64, json as _json, re, traceback, time, subprocess
import nest_asyncio, uvicorn, torch, requests as _req
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from typing import Optional
from PIL import Image

subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
os.system("pkill -f uvicorn 2>/dev/null || true")
time.sleep(1)
print("Port 8000 cleared")

nest_asyncio.apply()

app = FastAPI(title="ÉLUME Hair Studio API", version="3.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

# ── Helpers ────────────────────────────────────────────────────────────────
def _img_to_b64(img: Image.Image, quality: int = 92) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode()

def _strip_json(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    s = text.find("{")
    e = text.rfind("}")
    if s >= 0 and e > s:
        return text[s:e+1]
    return text.strip()

def _strip_json_array(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    s = text.find("[")
    e = text.rfind("]")
    if s >= 0 and e > s:
        return text[s:e+1]
    return text.strip()

def _vram_stats() -> dict:
    if not torch.cuda.is_available():
        return {}
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    return {"vram_used_gb": round(used, 2), "vram_total_gb": round(total, 2)}

# ── GET /health ────────────────────────────────────────────────────────────
@app.get("/health")
async def health():
    return {"status": "ok", "gpu": torch.cuda.is_available(),
            "device": DEVICE, **_vram_stats()}

# ── POST /mask_preview ─────────────────────────────────────────────────────
@app.post("/mask_preview")
async def mask_preview(image: UploadFile = File(...)):
    try:
        photo   = load_image(await image.read())
        mask    = get_hair_mask(photo)
        overlay = photo.copy().convert("RGBA")
        tint    = Image.new("RGBA", photo.size, (220, 60, 120, 110))
        overlay.paste(tint, mask=mask)
        return JSONResponse({
            "mask":    _img_to_b64(mask.convert("RGB")),
            "overlay": _img_to_b64(overlay.convert("RGB")),
        })
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /analyse ──────────────────────────────────────────────────────────
_ANALYSE_SYSTEM = (
    "You are a world-class hair stylist with 20 years of luxury salon experience. "
    "You always respect the biomechanical constraints provided. "
    "Each style you recommend MUST be visually distinct from the others – "
    "vary length, texture, formality, and structure. "
    "Return ONLY valid JSON. No markdown, no explanation."
)

def _build_analyse_prompt(profile, n, salon_styles, photo_present):
    constraint_block = profile_to_prompt_block(profile)
    salon_note = ""
    if salon_styles and salon_styles.strip():
        salon_note = (
            "SALON SIGNATURE STYLES (include ONE if it suits this face):\n"
            + salon_styles
            + "\nIf used, keep the exact style name and write a matching SDXL prompt."
        )
    photo_note = (
        "Also examine the uploaded photo for skin tone, current hair, and overall aesthetic."
        if photo_present else ""
    )
    style_rules = (
        "STYLE DIVERSITY RULES (mandatory):\n"
        "  Style 1: short cut (above chin)\n"
        "  Style 2: medium length (chin to collarbone)\n"
        "  Style 3: long style (past collarbone)\n"
        "  Style 4: textured/special technique (curly, braided, or sculpted)\n"
        "Every style must include the required SDXL modifiers from the face analysis."
    )
    return (
        f"{constraint_block}\n\n"
        f"{salon_note}\n\n"
        f"{photo_note}\n\n"
        f"{style_rules}\n\n"
        f"Recommend exactly {n} hairstyles that SATISFY the face constraints above.\n"
        "Return a JSON array of objects. Each object must have:\n"
        "  name          – 2-4 word elegant style name\n"
        "  why_this_face – one sentence citing the specific face metric that makes this work\n"
        "  sdxl_prompt   – 25-40 word SDXL inpainting prompt: include length, texture, "
                          "colour (natural brown unless salon specified), "
                          "AND the required face-specific SDXL modifiers\n"
        "  maintenance:\n"
        "    trim_weeks   – number (weeks between trims)\n"
        "    wash_days    – number (days between washes)\n"
        "    steps        – array of 3 specific styling steps\n"
        "    products     – array of 3 objects {name, type, reason}\n"
        "    grow_out_tip – one sentence\n"
    )

@app.post("/analyse")
async def analyse(
    face_shape:     str           = Form(...),
    face_length:    str           = Form(...),
    jaw_width:      str           = Form(...),
    forehead_width: str           = Form(...),
    cheek_width:    str           = Form(...),
    ratio:          str           = Form(...),
    count:          str           = Form("4"),
    salon_styles:   Optional[str] = Form(None),
    photo_b64:      Optional[str] = Form(None),
):
    try:
        n = min(max(int(count) if count.isdigit() else 4, 2), 6)
        profile  = build_face_profile(
            face_shape, face_length, jaw_width,
            forehead_width, cheek_width, ratio
        )
        user_text = _build_analyse_prompt(
            profile, n, salon_styles,
            photo_present=bool(photo_b64 and len(photo_b64) > 200)
        )
        use_vision = bool(photo_b64 and len(photo_b64) > 200)
        if use_vision:
            messages = [{"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{photo_b64}"}},
                {"type": "text", "text": user_text},
            ]}]
            model = "meta-llama/llama-4-scout-17b-16e-instruct"
        else:
            messages = [{"role": "user", "content": user_text}]
            model = "llama-3.3-70b-versatile"
        t0 = time.time()
        resp = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": _ANALYSE_SYSTEM}] + messages,
            temperature=0.75,
            max_tokens=3000,
        )
        elapsed = round(time.time() - t0, 2)
        raw    = resp.choices[0].message.content
        styles = _json.loads(_strip_json_array(raw))
        print(f"Analyse: {len(styles)} styles for {face_shape} | {elapsed}s | {model}")
        return JSONResponse({
            "styles":    styles,
            "profile":   profile,
            "model":     model,
            "elapsed_s": elapsed,
            "vision":    use_vision,
        })
    except _json.JSONDecodeError as e:
        raise HTTPException(status_code=422, detail=f"Invalid JSON from model: {e}")
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /generate ─────────────────────────────────────────────────────────
@app.post("/generate")
async def generate(
    image:           UploadFile           = File(...),
    custom_prompt:   str                  = Form(...),
    seed:            int                  = Form(42),
    steps:           int                  = Form(30),
    guidance:        float                = Form(8.0),
    strength:        float                = Form(0.95),
    dilation:        int                  = Form(28),
    reference_image: Optional[UploadFile] = File(None),
):
    try:
        if not custom_prompt or len(custom_prompt.strip()) < 5:
            raise HTTPException(status_code=400, detail="custom_prompt is required")
        steps    = max(10, min(steps, 50))
        guidance = max(1.0, min(guidance, 15.0))
        strength = max(0.5, min(strength, 1.0))
        dilation = max(10, min(dilation, 60))
        photo   = load_image(await image.read())
        ref_img = None
        if reference_image:
            try:
                ref_img = load_image(await reference_image.read())
                print(f"Reference loaded: {reference_image.filename}")
            except Exception as e:
                print(f"Reference load warning: {e}")
        t0 = time.time()
        result, mask = generate_hair(
            photo, custom_prompt.strip(),
            seed=seed, steps=steps, guidance=guidance,
            strength=strength, dilation=dilation,
            reference_image=ref_img,
        )
        elapsed = round(time.time() - t0, 2)
        print(f"Generate done in {elapsed}s")
        return JSONResponse({
            "result":    _img_to_b64(result),
            "original":  _img_to_b64(photo),
            "mask":      _img_to_b64(mask.convert("RGB")),
            "elapsed_s": elapsed,
            "params":    {"seed": seed, "steps": steps,
                          "guidance": guidance, "strength": strength},
        })
    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /describe_style ───────────────────────────────────────────────────
@app.post("/describe_style")
async def describe_style(
    style_image: Optional[UploadFile] = File(None),
    description: Optional[str]        = Form(None),
):
    try:
        desc = (description or "").strip()
        if style_image:
            raw = await style_image.read()
            b64 = base64.b64encode(raw).decode()
            extra = f' The client described it as: "{desc}".' if desc else ""
            resp = groq_client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[{"role": "user", "content": [
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                    {"type": "text", "text": (
                        "Describe this hairstyle as a professional stylist." + extra +
                        ' Return JSON only: {"description": "...", "sdxl_prompt": "..."}'
                    )},
                ]}],
                max_tokens=600,
            )
            return JSONResponse(_json.loads(_strip_json(resp.choices[0].message.content)))
        elif desc:
            raw = groq_text(
                f'Professional hairstylist. Client described: "{desc}". '
                f'Write a description and SDXL prompt. '
                f'JSON only: {{"description": "...", "sdxl_prompt": "..."}}',
                max_tokens=600
            )
            return JSONResponse(_json.loads(_strip_json(raw)))
        else:
            raise HTTPException(status_code=400, detail="Provide image or description")
    except _json.JSONDecodeError as e:
        raise HTTPException(status_code=422, detail=f"Invalid JSON from model: {e}")
    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# ── POST /consult ──────────────────────────────────────────────────────────
@app.post("/consult")
async def consult(
    style_name:  str = Form(...),
    face_shape:  str = Form(...),
    jaw_width:   str = Form("1.0"),
    fore_width:  str = Form("1.0"),
    cheek_width: str = Form("1.0"),
    ratio:       str = Form("1.5"),
):
    try:
        profile = build_face_profile(face_shape, "1.5", jaw_width, fore_width, cheek_width, ratio)
        avoid_str   = ", ".join(profile.get("avoid", [])[:3]) or "none"
        embrace_str = ", ".join(profile.get("embrace", [])[:3]) or "see below"
        raw = groq_text(
            f"You are a luxury salon beautician. Style: {style_name}. "
            f"Client face shape: {face_shape} "
            f"(avoid: {avoid_str}; embrace: {embrace_str}). "
            f"Return ONLY valid JSON (no markdown):\n"
            '{"maintenance_title": "...", '
            '"maintenance_steps": ["step1","step2","step3","step4","step5"], '
            '"products": ['
            '{"type":"Shampoo","name":"...","why":"..."}, '
            '{"type":"Conditioner","name":"...","why":"..."}, '
            '{"type":"Styling","name":"...","why":"..."}, '
            '{"type":"Treatment","name":"...","why":"..."}], '
            '"schedule": ['
            '{"label":"Wash frequency","value":"..."}, '
            '{"label":"Deep condition","value":"..."}, '
            '{"label":"Heat styling","value":"..."}, '
            '{"label":"Trim","value":"..."}], '
            '"salon_note": "...", "salon_return": "Every N weeks"}',
            max_tokens=900
        )
        return JSONResponse(_json.loads(_strip_json(raw)))
    except _json.JSONDecodeError as e:
        raise HTTPException(status_code=422, detail=f"Invalid JSON: {e}")
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

print("FastAPI ready – 6 endpoints:")
print("  GET  /health")
print("  POST /mask_preview  /analyse  /generate")
print("  POST /describe_style  /consult")


In [ ]:
# Cell 8 – START SERVER  ⚠️  Keep this cell running during demo
from pyngrok import ngrok, conf
import uvicorn, time, requests as _req

# Clear any stale ngrok tunnels
try:
    _h   = {"Authorization": f"Bearer {NGROK_TOKEN}", "Ngrok-Version": "2"}
    _eps = _req.get("https://api.ngrok.com/endpoints", headers=_h, timeout=5).json()
    for ep in _eps.get("endpoints", []):
        _req.delete(f"https://api.ngrok.com/endpoints/{ep['id']}", headers=_h)
        print(f"Cleared stale tunnel: {ep['public_url']}")
except Exception as _e:
    print(f"Could not clear endpoints (ok on first run): {_e}")

ngrok.kill()
time.sleep(2)
conf.get_default().auth_token = NGROK_TOKEN
ngrok.connect(addr=8000, proto="http", hostname=NGROK_DOMAIN)
PUBLIC_URL = f"https://{NGROK_DOMAIN}"

print()
print("=" * 62)
print("  ÉLUME GPU Backend  v3  –  LIVE")
print("=" * 62)
print(f"  URL            : {PUBLIC_URL}")
print(f"  Health         : {PUBLIC_URL}/health")
print(f"  Analyse        : {PUBLIC_URL}/analyse")
print(f"  Generate       : {PUBLIC_URL}/generate")
print(f"  Mask Preview   : {PUBLIC_URL}/mask_preview")
print(f"  Describe Style : {PUBLIC_URL}/describe_style")
print(f"  Consult        : {PUBLIC_URL}/consult")
print("=" * 62)
print("  URL is PERMANENT – no frontend changes needed")
print("  Restart only this cell when session expires")
print("=" * 62)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()
